In [4]:
from dataclasses import dataclass
from pathlib import Path
import pandas as pd

In [2]:
import os
os.chdir("..")
%pwd

'd:\\Projects\\RAG_Systems\\ProductDemand\\product-demand-forecasting-system'

In [5]:
@dataclass
class DataValidationConfig:
    root_dir: Path
    status_file: Path
    data_file: Path
    all_schema: dict

In [ ]:
import sys
from pathlib import Path
from src.productdemand.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH, SCHEMA_FILE_PATH
from src.productdemand.utils.common import read_yaml, create_directories
from src.productdemand.exception.custom_exception import CustomException



class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
        schema_filepath=SCHEMA_FILE_PATH,
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([Path(self.config.artifacts_root)])

    def get_data_validation_config(self) -> DataValidationConfig:
        config = self.config.data_validation
        schema = self.schema.COLUMNS

        create_directories([Path(config.root_dir)])

        data_validation_config = DataValidationConfig(
            root_dir=Path(config.root_dir),
            status_file=Path(config.STATUS_FILE),
            data_file=Path(self.config.data_ingestion.local_data_file),
            all_schema=dict(schema),
        )

        return data_validation_config

In [7]:
class DataValidation:
    def __init__(self, config: DataValidationConfig):
        self.config = config

    def validate_all_columns(self) -> bool:
        try:
            validation_status = True

            data = pd.read_csv(self.config.data_file, encoding="utf-8")
            all_cols = list(data.columns)
            all_schema_cols = list(self.config.all_schema.keys())

            missing_cols = [col for col in all_schema_cols if col not in all_cols]
            unexpected_cols = [col for col in all_cols if col not in all_schema_cols]

            if missing_cols or unexpected_cols:
                validation_status = False

            with open(self.config.status_file, "w", encoding="utf-8") as f:
                f.write(f"Validation status: {validation_status}\n")
                f.write(f"Missing columns: {missing_cols}\n")
                f.write(f"Unexpected columns: {unexpected_cols}\n")

            return validation_status

        except Exception as e:
            raise e

In [8]:
try:
    config = ConfigurationManager()
    data_validation_config = config.get_data_validation_config()
    data_validation = DataValidation(data_validation_config)
    result = data_validation.validate_all_columns()
    print("Validation Result:", result)
except Exception as e:
    raise e

{"path": "config\\config.yaml", "timestamp": "2026-06-24T13:14:19.525736Z", "level": "info", "event": "YAML file loaded successfully"}
{"path": "params.yaml", "timestamp": "2026-06-24T13:14:19.577097Z", "level": "info", "event": "YAML file loaded successfully"}
{"path": "schema.yaml", "timestamp": "2026-06-24T13:14:19.599886Z", "level": "info", "event": "YAML file loaded successfully"}
{"path": "artifacts", "timestamp": "2026-06-24T13:14:19.605971Z", "level": "info", "event": "Created directory"}
{"path": "artifacts\\data_validation", "timestamp": "2026-06-24T13:14:19.611028Z", "level": "info", "event": "Created directory"}


Validation Result: True
